# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file!")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
MODEL = "gemini-3.5-flash-lite"

API key found and looks good so far!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
#the user prompt is a function, unlike the system prompt, because it needs to be generated dynamically based on the URL being scraped
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [8]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.5-flash-lite
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'}]}

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 9 relevant links


{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-27B
Updated
8 days ago
•
2.09M
•
12.1k
unsloth/Qwen3.8-27B-GGUF
Updated
2 days ago
•
6.32M
•
2.58k
orcarouter/Qwen3.8-27B-Uncensored-FP8
Updated
2 days ago
•
143k
•
936
orcarouter/Qwen3.8-27B-Uncensored-MLX
Updated
1 day ago
•
34.9k
•
855
JonathanColetti/Qwen3.8

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\n8 days ago\n•\n2.09M\n•\n12.1k\nunsloth/Qwen3.8-27B-GGUF\nUpdated\n2 days ago\n•\n6.32M\n

In [16]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


# Hugging Face
## The AI Community Building the Future

Welcome to Hugging Face, the ultimate collaboration platform and home of machine learning. We empower developers, researchers, and organizations to create, discover, and build the future of artificial intelligence together. 

---

### What We Do
Hugging Face is the central hub where the global machine learning community comes together to collaborate on cutting-edge models, datasets, and applications. 

* **Over 2 Million Models:** Explore, share, and utilize state-of-the-art machine learning models ranging from language processing and audio to computer vision and advanced video generation.
* **Over 500k Datasets:** Access a vast repository of open data to train and fine-tune your models effectively.
* **Over 1 Million Spaces & AI Apps:** Discover, test, and deploy interactive machine learning applications, agents, and leaderboards directly in your browser.
* **Enterprise Solutions:** We offer robust infrastructure for businesses, including Team & Enterprise plans, Hugging Face PRO, enterprise support, inference providers, inference endpoints, and secure storage buckets.

---

### Our Customers
Our platform serves a diverse and rapidly growing global ecosystem. From independent developers and open-source contributors to world-class research institutions and enterprise organizations, Hugging Face provides the tools, hardware integrations, and scalable infrastructure needed to bring advanced AI projects from concept to production.

---

### Company Culture & Community
At the heart of Hugging Face is an open, vibrant, and collaborative community. We believe in the power of open-source AI and transparent research. 
* **Collaborative Spirit:** Our community actively contributes daily papers, community articles, and shared models to push the boundaries of technology.
* **Continuous Learning:** Through platforms like Discord, forums, GitHub, and educational tracks (such as Hugging Face Fundamentals), we foster an environment of continuous growth, shared knowledge, and inclusivity.
* **Cutting-Edge Innovation:** We stay at the forefront of the AI landscape—exploring agentic reinforcement learning, novel model architectures, and ethical AI development.

---

### Careers & Joining Us
Are you passionate about machine learning, open-source software, and building the infrastructure that powers the next generation of AI? 

Joining Hugging Face means working alongside top-tier engineers, researchers, and community builders who are shaping the global AI ecosystem. Whether your expertise lies in model architecture, distributed systems, developer relations, or AI ethics, you will find a dynamic, forward-thinking environment where your work directly impacts millions of developers worldwide. 

*Explore our platform, join our community on GitHub and Discord, and help us build the future of AI.*

# Adding a translator to the brochure.


In [24]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    return result
    

In [22]:
def translator_system_prompt(language): 
    system_prompt = f"""
You are a professional translator. You will be given a company brochure
written in markdown. Translate it into {language}, preserving markdown
formatting (headers, bullets, bold) and tone. Do not add commentary.

"""
    return system_prompt

In [23]:
def get_translator_user_prompt(brochure, company, language):
    user_prompt = f"""
Translate the following company brochure for {company} into {language}.
Preserve all markdown formatting exactly. Do not add any extra commentary,
notes, or explanations — output only the translated brochure.

Brochure:

{brochure}
"""


    return user_prompt

In [28]:
def translate_brochure(brochure, company_name, language):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": translator_system_prompt(language)},
            {"role": "user", "content": get_translator_user_prompt(brochure, company_name, language)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    return result


In [29]:
brochure = create_brochure("HuggingFace", "https://huggingface.co")
translated = translate_brochure(brochure, "HuggingFace", "French")


Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


# Hugging Face
## The AI Community Building the Future

Welcome to Hugging Face, the ultimate home and collaboration platform for machine learning. We bring together developers, researchers, and enthusiasts from all over the world to build, share, and scale the future of artificial intelligence.

---

### What We Offer

Hugging Face is the central hub where the ML community collaborates on cutting-edge models, datasets, and applications. 

* **Over 2 Million Models:** Discover, fine-tune, and deploy state-of-the-art machine learning models—from powerful text and language processors to advanced video and music generation tools.
* **Over 500 Thousand Datasets:** Access a vast library of open data to train and refine your AI projects.
* **Interactive Spaces:** Explore over 1 million live AI applications, demos, and leaderboards running directly on the platform.
* **Enterprise Solutions:** Scale your team's capabilities with Hugging Face PRO, dedicated enterprise support, custom inference providers, inference endpoints, and secure storage buckets.

---

### Our Community & Culture

At the heart of Hugging Face is an open, vibrant global community. We believe that the best AI is built together. Whether you are discussing the latest daily papers, collaborating through GitHub and Discord, or sharing your newest AI app, our platform thrives on shared knowledge, open science, and collective innovation.

---

### Join Us

Are you passionate about artificial intelligence and machine learning? Come build the future with us! 

* **For Customers:** Explore our flexible pricing plans, spin up enterprise-grade infrastructure, and accelerate your AI development cycle today.
* **For Investors:** Partner with the definitive platform driving the machine learning revolution and empowering millions of developers worldwide.
* **For Recruits:** Join a forward-thinking team at the epicenter of the AI boom. Collaborate with brilliant minds and help shape the tools that power the next generation of technology. 

Sign up today and become part of the AI community building the future!

# Hugging Face
## La communauté de l'IA qui bâtit l'avenir

Bienvenue chez Hugging Face, la plateforme ultime de collaboration et d'accueil pour l'apprentissage automatique (machine learning). Nous rassemblons des développeurs, des chercheurs et des passionnés du monde entier pour construire, partager et propulser l'avenir de l'intelligence artificielle.

---

### Ce que nous offrons

Hugging Face est le carrefour central où la communauté du machine learning collabore sur des modèles, des ensembles de données et des applications de pointe. 

* **Plus de 2 millions de modèles :** Découvrez, affinez et déployez des modèles d'apprentissage automatique de pointe, allant de puissants processeurs de texte et de langage à des outils avancés de génération de vidéo et de musique.
* **Plus de 500 mille ensembles de données :** Accédez à une vaste bibliothèque de données ouvertes pour entraîner et affiner vos projets d'IA.
* **Espaces interactifs :** Explorez plus d'un million d'applications d'IA en direct, de démonstrations et de classements fonctionnant directement sur la plateforme.
* **Solutions d'entreprise :** Développez les capacités de votre équipe avec Hugging Face PRO, un support d'entreprise dédié, des fournisseurs d'inférence personnalisés, des points de terminaison d'inférence et des espaces de stockage sécurisés.

---

### Notre communauté et notre culture

Au cœur de Hugging Face se trouve une communauté mondiale ouverte et dynamique. Nous croyons que la meilleure IA se construit ensemble. Que vous discutiez des derniers articles du jour, collaboriez via GitHub et Discord ou partagiez votre nouvelle application d'IA, notre plateforme prospère grâce au savoir partagé, à la science ouverte et à l'innovation collective.

---

### Rejoignez-nous

Êtes-vous passionné par l'intelligence artificielle et l'apprentissage automatique ? Venez construire l'avenir avec nous ! 

* **Pour les clients :** Découvrez nos plans tarifaires flexibles, déployez une infrastructure de niveau entreprise et accélérez votre cycle de développement en IA dès aujourd'hui.
* **Pour les investisseurs :** Devenez partenaire de la plateforme incontournable qui propulse la révolution du machine learning et habilite des millions de développeurs à travers le monde.
* **Pour les candidats :** Rejoignez une équipe avant-gardiste située à l'épicentre du boom de l'IA. Collaborez avec des esprits brillants et contribuez à façonner les outils qui alimentent la prochaine génération de technologie. 

Inscrivez-vous dès aujourd'hui et faites partie de la communauté de l'IA qui bâtit l'avenir !

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>